# **DermaSense: Facial Skin Disease Detection & Acne Severity Estimation**

**Objective:**
1. **Stage 1 (Diagnosis):** Detecting various facial skin diseases (Eczema, Melanoma, Basal Cell Carcinoma, etc.).
2. **Stage 2 (Severity):** Estimating the severity level of Acne (0-3) if detected.

**Methodology:**
- **Baseline Model:** Custom Convolutional Neural Network (CNN) trained from scratch.
- **Transfer Learning Backbone:** EfficientNet-B0 pretrained on ImageNet.
- **Hybrid Models:** Feature extraction with PCA + Classical Classifiers (SVM, Logistic Regression, MLP).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.decomposition import PCA

# Classifiers
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# PyTorch & Torchvision
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim.lr_scheduler import ReduceLROnPlateau

# --- REPRODUCIBILITY SETUP ---
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Configure Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation Device: {device}")

# Install Kagglehub
try:
    import kagglehub
except ImportError:
    os.system("pip install kagglehub")
    import kagglehub

plt.style.use('seaborn-v0_8-whitegrid')

# ==========================================
# 2. DATA ACQUISITION, PREPROCESSING & VISUALIZATION
# ==========================================

print("Downloading Datasets...")
path_skin = kagglehub.dataset_download("ismailpromus/skin-diseases-image-dataset")
path_acne = kagglehub.dataset_download("jincyjis/acne04")

SKIN_BASE = os.path.join(path_skin, "IMG_CLASSES")
ACNE_BASE = path_acne
for f in ["JPEGImages", "acne04", ".", "Images"]:
    t = os.path.join(path_acne, f)
    if os.path.exists(t) and len([x for x in os.listdir(t) if x.lower().endswith(".jpg")]) > 10:
        ACNE_BASE = t
        break

# --- Stage 1: Build DataFrame ---
rows_s1 = []
TARGETS = {
    "1. Eczema": "Eczema", "2. Melanoma": "Melanoma", "3. Atopic": "Eczema",
    "4. Basal": "Basal Cell Carcinoma", "5. Melanocytic": "Melanocytic Nevi",
    "6. Benign": "Benign Keratosis", "7. Psoriasis": "Psoriasis",
    "8. Seborrheic": "Seborrheic Keratoses"
}
MAX_IMAGES = 1600

if os.path.exists(SKIN_BASE):
    for d in os.listdir(SKIN_BASE):
        label = next((v for k, v in TARGETS.items() if d.startswith(k)), None)
        if label:
            folder = os.path.join(SKIN_BASE, d)
            imgs = [x for x in os.listdir(folder) if x.lower().endswith((".jpg", ".png"))][:MAX_IMAGES]
            for img in imgs:
                rows_s1.append({"path": os.path.join(folder, img), "label": label})

# Add Acne Class
if os.path.exists(ACNE_BASE):
    for img in [x for x in os.listdir(ACNE_BASE) if x.lower().endswith(".jpg")]:
        try:
            sev = int(img.split("_")[0].replace("levle", ""))
            label = "Clear_AlmostClear" if sev == 0 else "Acne"
            rows_s1.append({"path": os.path.join(ACNE_BASE, img), "label": label})
        except: continue

df_s1 = shuffle(pd.DataFrame(rows_s1), random_state=SEED).reset_index(drop=True)

# --- Stage 2: Build DataFrame ---
rows_s2 = []
if os.path.exists(ACNE_BASE):
    for img in [x for x in os.listdir(ACNE_BASE) if x.lower().endswith(".jpg")]:
        try:
            sev = int(img.split("_")[0].replace("levle", ""))
            rows_s2.append({"path": os.path.join(ACNE_BASE, img), "label": sev})
        except: continue

df_s2 = shuffle(pd.DataFrame(rows_s2), random_state=SEED).reset_index(drop=True)

# --- VISUALIZATION ---
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
df_s1['label'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title(f"Stage 1 Distribution (Total: {len(df_s1)})")
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 2, 2)
counts_s2 = df_s2['label'].value_counts().sort_index()
counts_s2.plot(kind='bar', color='salmon', edgecolor='black')
plt.title(f"Stage 2 Severity Distribution (Total: {len(df_s2)})")
plt.xlabel("Severity Level")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# ==========================================
# 3. DATASETS & DATALOADERS
# ==========================================

s1_classes = sorted(df_s1["label"].unique())
class_to_idx_s1 = {cls: i for i, cls in enumerate(s1_classes)}
s2_classes = ["Level 0", "Level 1", "Level 2", "Level 3"]

# Transforms
train_transforms_s1 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_transforms_s2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class SkinDataset(Dataset):
    def __init__(self, df, transform=None, stage=1):
        self.df = df
        self.transform = transform
        self.stage = stage
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
            if self.transform: img = self.transform(img)
            target = class_to_idx_s1[row["label"]] if self.stage == 1 else int(row["label"])
            return img, torch.tensor(target, dtype=torch.long)
        except: return self.__getitem__((idx + 1) % len(self.df))

# Loaders
BATCH_SIZE = 32
dataloaders = {}

# S1 Split
s1_train, s1_test = train_test_split(df_s1, test_size=0.15, stratify=df_s1["label"], random_state=SEED)
s1_train, s1_val = train_test_split(s1_train, test_size=0.1765, stratify=s1_train["label"], random_state=SEED)

# S2 Split
s2_train, s2_test = train_test_split(df_s2, test_size=0.15, stratify=df_s2["label"], random_state=SEED)
s2_train, s2_val = train_test_split(s2_train, test_size=0.1765, stratify=s2_train["label"], random_state=SEED)

# Weighted Sampler for Stage 2
targets = s2_train["label"].values
classes, class_counts = np.unique(targets, return_counts=True)
weight_per_class = {cls: 1.0/count for cls, count in zip(classes, class_counts)}
samples_weight = np.array([weight_per_class[t] for t in targets])
sampler = WeightedRandomSampler(torch.from_numpy(samples_weight).double(), len(samples_weight))

dataloaders['s1_train'] = DataLoader(SkinDataset(s1_train, train_transforms_s1, 1), batch_size=BATCH_SIZE, shuffle=True)
dataloaders['s1_val'] = DataLoader(SkinDataset(s1_val, val_transforms, 1), batch_size=BATCH_SIZE, shuffle=False)
dataloaders['s1_test'] = DataLoader(SkinDataset(s1_test, val_transforms, 1), batch_size=BATCH_SIZE, shuffle=False)

dataloaders['s2_train'] = DataLoader(SkinDataset(s2_train, train_transforms_s2, 2), batch_size=BATCH_SIZE, sampler=sampler)
dataloaders['s2_val'] = DataLoader(SkinDataset(s2_val, val_transforms, 2), batch_size=BATCH_SIZE, shuffle=False)
dataloaders['s2_test'] = DataLoader(SkinDataset(s2_test, val_transforms, 2), batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. MODEL ARCHITECTURES & TRAINING LOGIC
# ==========================================

# A. Custom CNN (Baseline)
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# Generic Training Function
def train_network(model, loaders, stage_key, epochs=25, patience=5):
    print(f"Starting training (Patience={patience})...")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        train_loss_epoch = 0.0

        for x, y in loaders[f'{stage_key}_train']:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            train_loss_epoch += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)

        epoch_train_acc = correct / total
        epoch_train_loss = train_loss_epoch / total

        model.eval()
        v_correct, v_total = 0, 0
        val_loss_epoch = 0.0
        with torch.no_grad():
            for x, y in loaders[f'{stage_key}_val']:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                val_loss_epoch += loss.item() * x.size(0)
                v_correct += (out.argmax(1) == y).sum().item()
                v_total += x.size(0)

        epoch_val_acc = v_correct / v_total
        epoch_val_loss = val_loss_epoch / v_total

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        scheduler.step(epoch_val_acc)

        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            best_wts = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early Stopping triggered.")
                break

    model.load_state_dict(best_wts)
    return model, best_acc, history

# Correct Test Evaluation Function
def evaluate_on_test(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return correct / total

# Hybrid Experiments Function
def run_hybrid_experiments(backbone, loaders, stage_key):
    print(f"\nRunning Hybrid Experiments for {stage_key}...")
    backbone.eval()

    def extract(loader):
        feats, lbls = [], []
        orig_head = backbone.classifier[1]
        backbone.classifier[1] = nn.Identity()
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                feats.append(backbone(x).cpu().numpy())
                lbls.append(y.numpy())
        backbone.classifier[1] = orig_head
        return np.vstack(feats), np.hstack(lbls)

    print("  Extracting features...")
    X_train, y_train = extract(loaders[f'{stage_key}_train'])
    X_val, y_val = extract(loaders[f'{stage_key}_val'])
    X_test, y_test = extract(loaders[f'{stage_key}_test'])

    X_full = np.vstack([X_train, X_val])
    y_full = np.hstack([y_train, y_val])

    pca = PCA(n_components=0.95, random_state=SEED)
    X_full_pca = pca.fit_transform(X_full)
    X_test_pca = pca.transform(X_test)

    classifiers = {
        "Logistic Regression": LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, class_weight='balanced'),
        "SVM": SVC(C=1.0, kernel='rbf', gamma='scale', class_weight='balanced'),
        "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, early_stopping=True, alpha=0.0001)
    }

    results = {}
    best_acc = 0
    best_model_name = ""
    best_preds = None

    for name, clf in classifiers.items():
        clf.fit(X_full_pca, y_full)
        y_pred = clf.predict(X_test_pca)
        acc = accuracy_score(y_test, y_pred)
        results[name] = acc
        if acc > best_acc:
            best_acc = acc
            best_model_name = name
            best_preds = y_pred

    return results, y_test, best_preds, best_model_name

# ==========================================
# 5. EXPERIMENTAL EXECUTION
# ==========================================

# --- 1. Baseline: Custom CNN ---
print("\n--- Baseline: Custom CNN (Stage 2) ---")
cnn_base = CustomCNN(num_classes=4).to(device)
# Train on val, but evaluate on TEST for fair comparison
cnn_base, _, _ = train_network(cnn_base, dataloaders, 's2', epochs=25, patience=5)
acc_baseline_test = evaluate_on_test(cnn_base, dataloaders['s2_test'])
print(f"Baseline CNN Test Accuracy: {acc_baseline_test:.4f}")

# --- 2. Backbone: EfficientNet ---
# Stage 1
print("\n--- Backbone: EfficientNet-B0 (Stage 1) ---")
eff_s1 = models.efficientnet_b0(weights='IMAGENET1K_V1')
eff_s1.classifier[1] = nn.Linear(1280, len(s1_classes))
eff_s1 = eff_s1.to(device)
eff_s1, _, _ = train_network(eff_s1, dataloaders, 's1', epochs=25, patience=5)
acc_effnet_s1_test = evaluate_on_test(eff_s1, dataloaders['s1_test'])

# Stage 2
print("\n--- Backbone: EfficientNet-B0 (Stage 2) ---")
eff_s2 = models.efficientnet_b0(weights='IMAGENET1K_V1')
eff_s2.classifier[1] = nn.Linear(1280, 4)
eff_s2 = eff_s2.to(device)
eff_s2, _, history_s2 = train_network(eff_s2, dataloaders, 's2', epochs=25, patience=5)
acc_effnet_s2_test = evaluate_on_test(eff_s2, dataloaders['s2_test'])
print(f"EfficientNet Stage 2 Test Accuracy: {acc_effnet_s2_test:.4f}")

# --- 3. Hybrid Experiments ---
print("\n--- Hybrid Experiments (Stage 1) ---")
res_s1, y_true_s1, y_pred_s1, best_name_s1 = run_hybrid_experiments(eff_s1, dataloaders, 's1')

print("\n--- Hybrid Experiments (Stage 2) ---")
res_s2, y_true_s2, y_pred_s2, best_name_s2 = run_hybrid_experiments(eff_s2, dataloaders, 's2')

# ==========================================
# 6. RESULTS & DISCUSSION
# ==========================================

def plot_cm(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    print(classification_report(y_true, y_pred, target_names=classes))

def plot_training_curves(history, title="Training Curves"):
    epochs = range(1, len(history['train_acc']) + 1)

    plt.figure(figsize=(14, 5))

    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'g-o', label='Val Acc')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_loss'], 'r-o', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'orange', marker='o', label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

# --- Visualizing Training Behavior ---
if history_s2['train_loss']:
    plot_training_curves(history_s2, title="EfficientNet Stage 2")

# --- Stage 1 Final Results ---
print(f"EfficientNet-B0 (Standard) Test Accuracy: {acc_effnet_s1_test:.4f}")
print(f"Best Hybrid Model (Stage 1): {best_name_s1} with Accuracy: {res_s1[best_name_s1]:.4f}")
plot_cm(y_true_s1, y_pred_s1, s1_classes, f"Stage 1: {best_name_s1}")

# --- Stage 2 Final Results ---
print(f"Best Hybrid Model (Stage 2): {best_name_s2} with Accuracy: {res_s2[best_name_s2]:.4f}")
plot_cm(y_true_s2, y_pred_s2, s2_classes, f"Stage 2: {best_name_s2}")

# --- Final Comparison Plot (All on Test Set) ---
plt.figure(figsize=(10, 6))

# Prepare fair data (All evaluated on Test Set)
models_list = ['Custom CNN (Baseline)', 'EfficientNet (Softmax)'] + list(res_s2.keys())
accuracies = [acc_baseline_test, acc_effnet_s2_test] + list(res_s2.values())

# Colors
colors = ['grey' if x < max(accuracies) else '#2ca02c' for x in accuracies]

bars = plt.bar(models_list, accuracies, color=colors)
plt.title("Comparative Analysis: Stage 2 Models (Test Set Accuracy)")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=45)

for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Project Pipeline Completed.")

torch.save(eff_s2.state_dict(), "stage2_effnet_current.pth")
torch.save(eff_s1.state_dict(), "stage1_effnet_current.pth")

# ==========================================
# Interactive Demo: Gradio UI for Stage 1 + Stage 2
# Uses the final trained EfficientNet weights so the UI can run
# directly without retraining.
# ==========================================

!pip install -q gradio
!git clone -q https://github.com/Bariqa1/skin-condition-recognition-model-weights.git

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import gradio as gr
from torchvision import transforms, models

# Device
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device in demo:", device)

# Paths to final trained weight files (loaded instead of retraining)
WEIGHT_S1_PATH = "/content/skin-condition-recognition-model-weights/efficientnet_s1_weights.pth"
WEIGHT_S2_PATH = "/content/skin-condition-recognition-model-weights/efficientnet_s2_weights.pth"

print("Exists S1:", os.path.exists(WEIGHT_S1_PATH))
print("Exists S2:", os.path.exists(WEIGHT_S2_PATH))

# Validation transforms (same preprocessing used during training)
if "val_transforms" not in globals():
    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])

# Stage 1 class names (fixed mapping used in training and evaluation)
if "s1_classes" not in globals():
    s1_classes = [
        "Acne",
        "Basal Cell Carcinoma",
        "Benign Keratosis",
        "Clear_AlmostClear",
        "Eczema",
        "Melanocytic Nevi",
        "Melanoma",
        "Psoriasis",
        "Seborrheic Keratoses"
    ]

# Build EfficientNet-B0 models and load the saved trained weights
eff_s1 = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
)
in_features_s1 = eff_s1.classifier[1].in_features
eff_s1.classifier[1] = nn.Linear(in_features_s1, len(s1_classes))
eff_s1.load_state_dict(torch.load(WEIGHT_S1_PATH, map_location=device))
eff_s1 = eff_s1.to(device)
eff_s1.eval()

eff_s2 = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
)
in_features_s2 = eff_s2.classifier[1].in_features
eff_s2.classifier[1] = nn.Linear(in_features_s2, 4)
eff_s2.load_state_dict(torch.load(WEIGHT_S2_PATH, map_location=device))
eff_s2 = eff_s2.to(device)
eff_s2.eval()

severity_labels = [
    "Level 0 (Clear / Almost Clear)",
    "Level 1 (Mild Acne)",
    "Level 2 (Moderate Acne)",
    "Level 3 (Severe Acne)"
]

# Inference pipeline: Stage 1 diagnosis -> Stage 2 severity (if Acne)
def predict_pipeline(img: Image.Image):
    if img is None:
        return "No image provided.", ""

    x = val_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits_s1 = eff_s1(x)
        probs_s1 = F.softmax(logits_s1, dim=1)[0].cpu().numpy()

    idx1 = int(probs_s1.argmax())
    disease_label = s1_classes[idx1]
    disease_prob = float(probs_s1[idx1])
    disease_text = f"Disease prediction: {disease_label} (prob = {disease_prob:.3f})"

    severity_text = "Severity estimation: Not applicable (non-acne case)."
    if disease_label == "Acne":
        with torch.no_grad():
            logits_s2 = eff_s2(x)
            probs_s2 = F.softmax(logits_s2, dim=1)[0].cpu().numpy()

        idx2 = int(probs_s2.argmax())
        severity_label = severity_labels[idx2]
        severity_prob = float(probs_s2[idx2])
        severity_text = f"Acne severity: {severity_label} (prob = {severity_prob:.3f})"

    return disease_text, severity_text

# Gradio demo for interactive testing
demo = gr.Interface(
    fn=predict_pipeline,
    inputs=gr.Image(type="pil", label="Upload face skin image"),
    outputs=[
        gr.Textbox(label="Stage 1 – Disease Diagnosis"),
        gr.Textbox(label="Stage 2 – Acne Severity (if Acne)")
    ],
    title="Facial Skin Disease & Acne Severity Estimation",
    description=(
        "Upload a facial skin image. "
        "The system predicts the disease class; if the diagnosis is Acne, "
        "it also estimates the severity level (0–3)."
    ),
)

demo.launch(share=True)